# 13A — Export Skeleton GaitTR Embeddings for Fusion — FULL FIXED

এই notebook আপনার uploaded training/evaluation script থেকে exact GaitTR architecture ব্যবহার করে skeleton embeddings export করবে।

Why needed:

```text
13_train_adaptive_fusion_gate_FULL_FIXED.ipynb
```

fusion training/evaluation-এর জন্য paired embeddings চায়:

```text
skeleton_train_LT_embeddings.npz
skeleton_gallery_LT_embeddings.npz
skeleton_probe_LT_nm_embeddings.npz
skeleton_probe_LT_bg_embeddings.npz
skeleton_probe_LT_cl_embeddings.npz
```

এই notebook এগুলো save করবে এখানে:

```text
results/fusion_gate/expert_embedding_cache/
```

Then rerun notebook 13 from top.

In [1]:
# ============================================================
# CELL 1 — Imports, paths, config
# ============================================================

from pathlib import Path
from contextlib import nullcontext
import json
import time
import gc
import warnings

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    from tqdm.notebook import tqdm
except Exception:
    try:
        from tqdm import tqdm
    except Exception:
        def tqdm(x, *args, **kwargs):
            return x

warnings.filterwarnings("ignore")

EXP_DIR = Path("/media/wadud/DriveUbuntu/GaitRecognition 2.0")
SPLIT_NAME = "LT"

FUSION_SPLIT_DIR = EXP_DIR / "data" / "fusion_splits"
CHECKPOINT_DIR = EXP_DIR / "checkpoints"
RESULT_DIR = EXP_DIR / "results"

FUSION_ROOT = RESULT_DIR / "fusion_gate"
CACHE_DIR = FUSION_ROOT / "expert_embedding_cache"
REPORT_DIR = FUSION_ROOT / "embedding_export_reports"

for d in [FUSION_ROOT, CACHE_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Best skeleton checkpoint according to your previous skeleton evaluation.
CHECKPOINT_PATH = CHECKPOINT_DIR / "gaittr_LT_ce_triplet_best_loss.pth"

# Alternative checkpoints:
# CHECKPOINT_PATH = CHECKPOINT_DIR / "gaittr_LT_ce_triplet_last.pth"
# CHECKPOINT_PATH = CHECKPOINT_DIR / "gaittr_LT_ce_triplet_best_train_acc.pth"
# CHECKPOINT_PATH = CHECKPOINT_DIR / "gaittr_LT_ce_triplet_full_last.pth"

TRAIN_CSV = FUSION_SPLIT_DIR / f"train_{SPLIT_NAME}_fusion.csv"
GALLERY_CSV = FUSION_SPLIT_DIR / f"gallery_{SPLIT_NAME}_fusion.csv"
PROBE_CSVS = {
    "NM": FUSION_SPLIT_DIR / f"probe_{SPLIT_NAME}_nm_fusion.csv",
    "BG": FUSION_SPLIT_DIR / f"probe_{SPLIT_NAME}_bg_fusion.csv",
    "CL": FUSION_SPLIT_DIR / f"probe_{SPLIT_NAME}_cl_fusion.csv",
}

# Must match your 06 evaluation notebook.
EVAL_MODE = "sliding"   # "sliding", "center", "full"
SEQ_LEN = 60
STRIDE = 30
BATCH_CLIPS = 128
USE_EVAL_AMP = True

# Run switches.
EXPORT_TRAIN = True
EXPORT_GALLERY = True
EXPORT_PROBES = True

# If True, skip already exported files.
SKIP_IF_EXISTS = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

print("=" * 80)
print("13A — Export Skeleton GaitTR Embeddings for Fusion")
print("=" * 80)
print("EXP_DIR        :", EXP_DIR)
print("CHECKPOINT_PATH:", CHECKPOINT_PATH)
print("FUSION_SPLIT_DIR:", FUSION_SPLIT_DIR)
print("CACHE_DIR      :", CACHE_DIR)
print("EVAL_MODE      :", EVAL_MODE)
print("Device         :", device)
if torch.cuda.is_available():
    print("GPU            :", torch.cuda.get_device_name(0))
print("=" * 80)

assert CHECKPOINT_PATH.exists(), f"Checkpoint not found: {CHECKPOINT_PATH}"
assert TRAIN_CSV.exists(), f"Missing: {TRAIN_CSV}"
assert GALLERY_CSV.exists(), f"Missing: {GALLERY_CSV}"
for k, p in PROBE_CSVS.items():
    assert p.exists(), f"Missing {k}: {p}"

13A — Export Skeleton GaitTR Embeddings for Fusion
EXP_DIR        : /media/wadud/DriveUbuntu/GaitRecognition 2.0
CHECKPOINT_PATH: /media/wadud/DriveUbuntu/GaitRecognition 2.0/checkpoints/gaittr_LT_ce_triplet_best_loss.pth
FUSION_SPLIT_DIR: /media/wadud/DriveUbuntu/GaitRecognition 2.0/data/fusion_splits
CACHE_DIR      : /media/wadud/DriveUbuntu/GaitRecognition 2.0/results/fusion_gate/expert_embedding_cache
EVAL_MODE      : sliding
Device         : cuda
GPU            : NVIDIA GeForce RTX 3060 Laptop GPU


In [2]:
# ============================================================
# CELL 2 — Load fusion split CSVs and sample keys
# ============================================================

csv_dtype = {
    "pose_path": str,
    "silhouette_path": str,
    "subject": str,
    "condition": str,
    "seq": str,
    "view": str,
}

def normalize_view(v):
    v = str(v)
    if v.isdigit():
        return f"{int(v):03d}"
    return v

def add_sample_key(df):
    df = df.copy()
    for c in ["subject", "condition", "seq", "view"]:
        df[c] = df[c].astype(str)
    df["view"] = df["view"].apply(normalize_view)
    df["sample_key"] = (
        df["subject"].astype(str) + "-" +
        df["condition"].astype(str) + "-" +
        df["seq"].astype(str) + "-" +
        df["view"].astype(str)
    )
    return df

def load_fusion_csv(path):
    df = pd.read_csv(path, dtype=csv_dtype)
    needed = ["pose_path", "subject", "condition", "seq", "view"]
    missing = [c for c in needed if c not in df.columns]
    assert not missing, f"{path} missing columns: {missing}"
    df = add_sample_key(df)

    missing_pose = [p for p in df["pose_path"].tolist() if not Path(p).exists()]
    if missing_pose:
        print("First missing pose:", missing_pose[0])
        raise FileNotFoundError(f"{path.name}: missing pose paths = {len(missing_pose)}")

    return df

df_train = load_fusion_csv(TRAIN_CSV)
df_gallery = load_fusion_csv(GALLERY_CSV)
probe_dfs = {name: load_fusion_csv(path) for name, path in PROBE_CSVS.items()}

print("Train rows  :", len(df_train), "subjects:", df_train["subject"].nunique())
print("Gallery rows:", len(df_gallery), "subjects:", df_gallery["subject"].nunique())
for name, df in probe_dfs.items():
    print(f"Probe {name} rows:", len(df), "subjects:", df["subject"].nunique())

display(df_train.head())

Train rows  : 8139 subjects: 74
Gallery rows: 2197 subjects: 50
Probe NM rows: 1100 subjects: 50
Probe BG rows: 1100 subjects: 50
Probe CL rows: 1100 subjects: 50


,split_pose_path,subject,condition,seq,view,pose_path,silhouette_path,T_pose,T_silhouette,T_common,T_diff,alignment_status,sil_valid_frame_ratio,sil_mean_det_score,sil_mean_mask_area,sample_key
0,/media/wadud/DriveUbuntu/GaitRecognition 2.0/d...,001,bg,01,000,/media/wadud/DriveUbuntu/GaitRecognition 2.0/d...,/media/wadud/DriveUbuntu/GaitRecognition 2.0/d...,99.0,99.0,99.0,0.0,exact,1.000000,0.910541,4039.515137,001-bg-01-000
1,/media/wadud/DriveUbuntu/GaitRecognition 2.0/d...,001,bg,01,018,/media/wadud/DriveUbuntu/GaitRecognition 2.0/d...,/media/wadud/DriveUbuntu/GaitRecognition 2.0/d...,102.0,102.0,102.0,0.0,exact,1.000000,0.905187,4417.048828,001-bg-01-018
2,/media/wadud/DriveUbuntu/GaitRecognition 2.0/d...,001,bg,01,036,/media/wadud/DriveUbuntu/GaitRecognition 2.0/d...,/media/wadud/DriveUbuntu/GaitRecognition 2.0/d...,99.0,99.0,99.0,0.0,exact,0.969697,0.862985,3496.373779,001-bg-01-036
3,/media/wadud/DriveUbuntu/GaitRecognition 2.0/d...,001,bg,01,054,/media/wadud/DriveUbuntu/GaitRecognition 2.0/d...,/media/wadud/DriveUbuntu/GaitRecognition 2.0/d...,106.0,106.0,106.0,0.0,exact,0.943396,0.839839,2664.773682,001-bg-01-054
4,/media/wadud/DriveUbuntu/GaitRecognition 2.0/d...,001,bg,01,072,/media/wadud/DriveUbuntu/GaitRecognition 2.0/d...,/media/wadud/DriveUbuntu/GaitRecognition 2.0/d...,106.0,106.0,106.0,0.0,exact,0.811321,0.682147,2239.179199,001-bg-01-072


In [3]:
# ============================================================
# CELL 3 — Feature builder from your exact 06 evaluation script
# ============================================================

COCO_PARENTS = np.array([
    0, 0, 0, 1, 2,
    0, 0, 5, 6, 7, 8,
    5, 6, 11, 12, 13, 14
], dtype=np.int64)

def crop_or_pad_sequence(X, seq_len=60, random_crop=False):
    T = X.shape[0]
    if T == seq_len:
        return X
    if T > seq_len:
        start = np.random.randint(0, T - seq_len + 1) if random_crop else max(0, (T - seq_len) // 2)
        return X[start:start + seq_len]
    pad_len = seq_len - T
    pad = np.repeat(X[-1:], pad_len, axis=0)
    return np.concatenate([X, pad], axis=0)

def make_sliding_clips(X, clip_len=60, stride=30):
    T = X.shape[0]
    if T <= clip_len:
        return [crop_or_pad_sequence(X, clip_len, random_crop=False)]

    clips = []
    for start in range(0, T - clip_len + 1, stride):
        clips.append(X[start:start + clip_len])

    last_start = T - clip_len
    if last_start > 0:
        tail_clip = X[last_start:last_start + clip_len]
        if len(clips) == 0 or not np.array_equal(clips[-1], tail_clip):
            clips.append(tail_clip)

    return clips

def build_gaittr_features(X):
    X = X.astype(np.float32)
    assert X.ndim == 3 and X.shape[1:] == (17, 2), f"Bad skeleton shape: {X.shape}"

    joint = X.copy()

    nose = X[:, 0:1, :]
    joint_rel = X - nose

    v1 = np.zeros_like(X)
    v1[:-1] = X[1:] - X[:-1]

    v2 = np.zeros_like(X)
    v2[:-2] = X[2:] - X[:-2]

    bone = X - X[:, COCO_PARENTS, :]

    # Final: C x T x V, C=10
    feat = np.concatenate([joint, joint_rel, v1, v2, bone], axis=-1)
    feat = feat.transpose(2, 0, 1).astype(np.float32)
    return feat

print("[OK] Feature builder loaded.")

[OK] Feature builder loaded.


In [4]:
# ============================================================
# CELL 4 — Exact GaitTR backbone from your 06 evaluation script
# ============================================================

class TCN(nn.Module):
    def __init__(self, in_channels, out_channels, temporal_kernel=9, dropout=0.1):
        super().__init__()
        pad = temporal_kernel // 2
        self.net = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=(temporal_kernel, 1),
                padding=(pad, 0),
                bias=False,
            ),
            nn.Dropout(dropout),
            nn.Mish(),
            nn.BatchNorm2d(out_channels),
        )

    def forward(self, x):
        return self.net(x)

class SpatialTransformer(nn.Module):
    def __init__(self, channels, num_heads=8, dropout=0.1):
        super().__init__()
        assert channels % num_heads == 0
        self.norm = nn.LayerNorm(channels)
        self.attn = nn.MultiheadAttention(
            channels,
            num_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.proj = nn.Linear(channels, channels)
        self.act = nn.Mish()
        self.bn = nn.BatchNorm2d(channels)

    def forward(self, x):
        B, C, T, V = x.shape

        tokens = x.permute(0, 2, 3, 1).contiguous().view(B * T, V, C)
        tokens = self.norm(tokens)

        out, _ = self.attn(tokens, tokens, tokens, need_weights=False)
        out = self.proj(out)

        out = out.view(B, T, V, C).permute(0, 3, 1, 2).contiguous()
        out = self.act(out)
        out = self.bn(out)
        return out

class TCNSTBlock(nn.Module):
    def __init__(self, in_channels, out_channels, num_heads=8, temporal_kernel=9, dropout=0.1):
        super().__init__()
        self.tcn = TCN(in_channels, out_channels, temporal_kernel, dropout)
        self.st = SpatialTransformer(out_channels, num_heads, dropout)

        if in_channels == out_channels:
            self.residual = nn.Identity()
        else:
            self.residual = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
                nn.Mish(),
                nn.BatchNorm2d(out_channels),
            )

    def forward(self, x):
        return self.st(self.tcn(x)) + self.residual(x)

class GaitTRBackbone(nn.Module):
    def __init__(
        self,
        in_channels=10,
        embedding_dim=128,
        channels=(64, 64, 128, 256),
        num_heads=8,
        temporal_kernel=9,
        dropout=0.1,
    ):
        super().__init__()

        self.data_bn = nn.BatchNorm2d(in_channels)

        blocks = []
        prev = in_channels
        for ch in channels:
            blocks.append(
                TCNSTBlock(
                    in_channels=prev,
                    out_channels=ch,
                    num_heads=num_heads,
                    temporal_kernel=temporal_kernel,
                    dropout=dropout,
                )
            )
            prev = ch

        self.blocks = nn.Sequential(*blocks)
        self.fc = nn.Linear(channels[-1], embedding_dim, bias=False)

    def forward(self, x):
        x = self.data_bn(x)
        x = self.blocks(x)
        x = x.mean(dim=(2, 3))
        emb = self.fc(x)
        emb = F.normalize(emb, p=2, dim=1)
        return emb

print("[OK] Exact GaitTRBackbone loaded.")

[OK] Exact GaitTRBackbone loaded.


In [5]:
# ============================================================
# CELL 5 — Load checkpoint robustly
# ============================================================

def safe_torch_load(path, map_location):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)

def strip_prefix_if_present(state_dict, prefix):
    if any(k.startswith(prefix) for k in state_dict.keys()):
        return {
            (k[len(prefix):] if k.startswith(prefix) else k): v
            for k, v in state_dict.items()
        }
    return state_dict

def filter_backbone_state_dict(state_dict):
    state_dict = strip_prefix_if_present(state_dict, "module.")
    state_dict = strip_prefix_if_present(state_dict, "model.")
    state_dict = strip_prefix_if_present(state_dict, "backbone.")

    filtered = {}
    removed = []
    for k, v in state_dict.items():
        if k.startswith("classifier."):
            removed.append(k)
            continue
        filtered[k] = v
    return filtered, removed

ckpt = safe_torch_load(CHECKPOINT_PATH, map_location=device)
config = ckpt.get("config", {})

in_channels = int(config.get("in_channels", 10))
embedding_dim = int(config.get("embedding_dim", 128))
channels = tuple(config.get("channels", [64, 64, 128, 256]))
num_heads = int(config.get("num_heads", 8))
temporal_kernel = int(config.get("temporal_kernel", 9))
dropout = float(config.get("dropout", 0.1))

raw_state = ckpt["model"]
state_dict, removed_keys = filter_backbone_state_dict(raw_state)

model = GaitTRBackbone(
    in_channels=in_channels,
    embedding_dim=embedding_dim,
    channels=channels,
    num_heads=num_heads,
    temporal_kernel=temporal_kernel,
    dropout=dropout,
).to(device)

missing_keys, unexpected_keys = model.load_state_dict(state_dict, strict=False)

print("checkpoint_type:", ckpt.get("checkpoint_type", "unknown"))
print("step           :", ckpt.get("step", "unknown"))
print("embedding_dim  :", embedding_dim)
print("channels       :", channels)
print("removed classifier keys:", len(removed_keys))
print("missing keys   :", missing_keys)
print("unexpected keys:", unexpected_keys)

if missing_keys:
    raise RuntimeError("Checkpoint load has missing keys. Stop to avoid invalid embeddings.")
if unexpected_keys:
    raise RuntimeError("Checkpoint load has unexpected keys. Stop to avoid invalid embeddings.")

model.eval()

with torch.no_grad():
    dummy = torch.randn(2, in_channels, SEQ_LEN, 17).to(device)
    emb = model(dummy)
print("Dummy embedding shape:", tuple(emb.shape))
assert emb.shape == (2, embedding_dim)

del dummy, emb
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("[OK] Checkpoint loaded safely.")

checkpoint_type: eval_backbone_only
step           : 29550
embedding_dim  : 128
channels       : (64, 64, 128, 256)
removed classifier keys: 0
missing keys   : []
unexpected keys: []
Dummy embedding shape: (2, 128)
[OK] Checkpoint loaded safely.


In [6]:
# ============================================================
# CELL 6 — Embedding extraction functions
# ============================================================

def eval_autocast_context():
    if device.type == "cuda" and USE_EVAL_AMP:
        try:
            return torch.amp.autocast("cuda", enabled=True)
        except Exception:
            return torch.cuda.amp.autocast(enabled=True)
    return nullcontext()

def load_pose_array(pose_path):
    data = np.load(pose_path)

    if "keypoints_norm_filled" in data.files:
        X = data["keypoints_norm_filled"].astype(np.float32)
    elif "keypoints_norm" in data.files:
        X = data["keypoints_norm"].astype(np.float32)
    elif "keypoints" in data.files:
        X = data["keypoints"].astype(np.float32)
    else:
        raise KeyError(f"No keypoints array found in {pose_path}. Keys={data.files}")

    if X.ndim == 2 and X.shape[1] == 34:
        X = X.reshape(X.shape[0], 17, 2)

    assert X.ndim == 3 and X.shape[1:] == (17, 2), f"Bad pose shape {X.shape} in {pose_path}"
    return X

@torch.no_grad()
def extract_video_embedding(pose_path):
    X = load_pose_array(pose_path)

    if EVAL_MODE == "full":
        feat = build_gaittr_features(X)
        x = torch.from_numpy(feat).unsqueeze(0).float().to(device)
        with eval_autocast_context():
            emb = model(x)
        emb = emb.squeeze(0).float().cpu()
        return F.normalize(emb, p=2, dim=0)

    if EVAL_MODE == "center":
        X60 = crop_or_pad_sequence(X, seq_len=SEQ_LEN, random_crop=False)
        feat = build_gaittr_features(X60)
        x = torch.from_numpy(feat).unsqueeze(0).float().to(device)
        with eval_autocast_context():
            emb = model(x)
        emb = emb.squeeze(0).float().cpu()
        return F.normalize(emb, p=2, dim=0)

    if EVAL_MODE == "sliding":
        clips = make_sliding_clips(X, clip_len=SEQ_LEN, stride=STRIDE)
        feats = np.stack([build_gaittr_features(c) for c in clips], axis=0)

        embs = []
        for start in range(0, len(feats), BATCH_CLIPS):
            batch = torch.from_numpy(feats[start:start + BATCH_CLIPS]).float().to(device)
            with eval_autocast_context():
                emb = model(batch)
            embs.append(emb.float().cpu())

        embs = torch.cat(embs, dim=0)
        final_emb = embs.mean(dim=0)
        return F.normalize(final_emb, p=2, dim=0)

    raise ValueError(f"Unknown EVAL_MODE: {EVAL_MODE}")

def output_paths(split_tag):
    npz_path = CACHE_DIR / f"skeleton_{split_tag}_embeddings.npz"
    csv_path = CACHE_DIR / f"skeleton_{split_tag}_meta.csv"
    pt_path = CACHE_DIR / f"skeleton_{split_tag}_embeddings.pt"
    return npz_path, csv_path, pt_path

def extract_embeddings_for_df(df, split_tag, display_name):
    npz_path, csv_path, pt_path = output_paths(split_tag)

    if SKIP_IF_EXISTS and npz_path.exists() and csv_path.exists():
        print(f"[SKIP] {display_name}: already exists")
        emb = np.load(npz_path)["embeddings"]
        meta = pd.read_csv(csv_path, dtype=str)
        return meta, torch.from_numpy(emb.astype(np.float32))

    rows = []
    embeddings = []

    start_time = time.time()

    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Extract {display_name}"):
        emb = extract_video_embedding(row["pose_path"])
        embeddings.append(emb)

        out_row = {
            "sample_key": str(row["sample_key"]),
            "subject": str(row["subject"]),
            "condition": str(row["condition"]),
            "seq": str(row["seq"]),
            "view": str(row["view"]),
            "pose_path": str(row["pose_path"]),
        }

        if "silhouette_path" in row.index:
            out_row["silhouette_path"] = str(row["silhouette_path"])

        rows.append(out_row)

    meta_df = add_sample_key(pd.DataFrame(rows))
    emb_tensor = torch.stack(embeddings, dim=0).float()
    emb_np = emb_tensor.cpu().numpy().astype(np.float32)

    np.savez_compressed(npz_path, embeddings=emb_np)
    meta_df.to_csv(csv_path, index=False)

    # Compatibility with previous skeleton eval style.
    torch.save({
        "meta": meta_df,
        "embeddings": emb_tensor,
        "checkpoint": str(CHECKPOINT_PATH),
        "eval_mode": EVAL_MODE,
        "split_tag": split_tag,
    }, pt_path)

    elapsed = time.time() - start_time

    print(f"{display_name}: embeddings={tuple(emb_tensor.shape)}, elapsed_min={elapsed/60:.2f}")
    print("Saved:", npz_path)
    print("Saved:", csv_path)
    print("Saved:", pt_path)

    return meta_df, emb_tensor

print("[OK] Extraction functions ready.")

[OK] Extraction functions ready.


In [7]:
# ============================================================
# CELL 7 — Export skeleton embeddings for fusion
# ============================================================

exported = {}

if EXPORT_TRAIN:
    meta_train, emb_train = extract_embeddings_for_df(
        df_train,
        split_tag=f"train_{SPLIT_NAME}",
        display_name=f"train_{SPLIT_NAME}",
    )
    exported["train"] = {"meta": meta_train, "emb": emb_train}

if EXPORT_GALLERY:
    meta_gallery, emb_gallery = extract_embeddings_for_df(
        df_gallery,
        split_tag=f"gallery_{SPLIT_NAME}",
        display_name=f"gallery_{SPLIT_NAME}",
    )
    exported["gallery"] = {"meta": meta_gallery, "emb": emb_gallery}

if EXPORT_PROBES:
    for cond, df_probe in probe_dfs.items():
        meta_probe, emb_probe = extract_embeddings_for_df(
            df_probe,
            split_tag=f"probe_{SPLIT_NAME}_{cond.lower()}",
            display_name=f"probe_{SPLIT_NAME}_{cond.lower()}",
        )
        exported[cond] = {"meta": meta_probe, "emb": emb_probe}

print("Export complete. Splits:", list(exported.keys()))

Extract train_LT:   0%|          | 0/8139 [00:00<?, ?it/s]

train_LT: embeddings=(8139, 128), elapsed_min=0.86
Saved: /media/wadud/DriveUbuntu/GaitRecognition 2.0/results/fusion_gate/expert_embedding_cache/skeleton_train_LT_embeddings.npz
Saved: /media/wadud/DriveUbuntu/GaitRecognition 2.0/results/fusion_gate/expert_embedding_cache/skeleton_train_LT_meta.csv
Saved: /media/wadud/DriveUbuntu/GaitRecognition 2.0/results/fusion_gate/expert_embedding_cache/skeleton_train_LT_embeddings.pt


Extract gallery_LT:   0%|          | 0/2197 [00:00<?, ?it/s]

gallery_LT: embeddings=(2197, 128), elapsed_min=0.19
Saved: /media/wadud/DriveUbuntu/GaitRecognition 2.0/results/fusion_gate/expert_embedding_cache/skeleton_gallery_LT_embeddings.npz
Saved: /media/wadud/DriveUbuntu/GaitRecognition 2.0/results/fusion_gate/expert_embedding_cache/skeleton_gallery_LT_meta.csv
Saved: /media/wadud/DriveUbuntu/GaitRecognition 2.0/results/fusion_gate/expert_embedding_cache/skeleton_gallery_LT_embeddings.pt


Extract probe_LT_nm:   0%|          | 0/1100 [00:00<?, ?it/s]

probe_LT_nm: embeddings=(1100, 128), elapsed_min=0.10
Saved: /media/wadud/DriveUbuntu/GaitRecognition 2.0/results/fusion_gate/expert_embedding_cache/skeleton_probe_LT_nm_embeddings.npz
Saved: /media/wadud/DriveUbuntu/GaitRecognition 2.0/results/fusion_gate/expert_embedding_cache/skeleton_probe_LT_nm_meta.csv
Saved: /media/wadud/DriveUbuntu/GaitRecognition 2.0/results/fusion_gate/expert_embedding_cache/skeleton_probe_LT_nm_embeddings.pt


Extract probe_LT_bg:   0%|          | 0/1100 [00:00<?, ?it/s]

probe_LT_bg: embeddings=(1100, 128), elapsed_min=0.10
Saved: /media/wadud/DriveUbuntu/GaitRecognition 2.0/results/fusion_gate/expert_embedding_cache/skeleton_probe_LT_bg_embeddings.npz
Saved: /media/wadud/DriveUbuntu/GaitRecognition 2.0/results/fusion_gate/expert_embedding_cache/skeleton_probe_LT_bg_meta.csv
Saved: /media/wadud/DriveUbuntu/GaitRecognition 2.0/results/fusion_gate/expert_embedding_cache/skeleton_probe_LT_bg_embeddings.pt


Extract probe_LT_cl:   0%|          | 0/1100 [00:00<?, ?it/s]

probe_LT_cl: embeddings=(1100, 128), elapsed_min=0.10
Saved: /media/wadud/DriveUbuntu/GaitRecognition 2.0/results/fusion_gate/expert_embedding_cache/skeleton_probe_LT_cl_embeddings.npz
Saved: /media/wadud/DriveUbuntu/GaitRecognition 2.0/results/fusion_gate/expert_embedding_cache/skeleton_probe_LT_cl_meta.csv
Saved: /media/wadud/DriveUbuntu/GaitRecognition 2.0/results/fusion_gate/expert_embedding_cache/skeleton_probe_LT_cl_embeddings.pt
Export complete. Splits: ['train', 'gallery', 'NM', 'BG', 'CL']


In [8]:
# ============================================================
# CELL 8 — Verify exported files and embedding health
# ============================================================

expected = {
    "train": len(df_train),
    "gallery": len(df_gallery),
    "NM": len(probe_dfs["NM"]),
    "BG": len(probe_dfs["BG"]),
    "CL": len(probe_dfs["CL"]),
}

verify_rows = []

split_tags = {
    "train": f"train_{SPLIT_NAME}",
    "gallery": f"gallery_{SPLIT_NAME}",
    "NM": f"probe_{SPLIT_NAME}_nm",
    "BG": f"probe_{SPLIT_NAME}_bg",
    "CL": f"probe_{SPLIT_NAME}_cl",
}

for name, split_tag in split_tags.items():
    npz_path, csv_path, pt_path = output_paths(split_tag)

    exists = npz_path.exists() and csv_path.exists()
    row = {
        "split": name,
        "split_tag": split_tag,
        "expected_rows": expected[name],
        "npz_exists": npz_path.exists(),
        "csv_exists": csv_path.exists(),
        "pt_exists": pt_path.exists(),
        "npz_path": str(npz_path),
        "csv_path": str(csv_path),
    }

    if exists:
        emb = np.load(npz_path)["embeddings"]
        meta = pd.read_csv(csv_path, dtype=str)
        norms = np.linalg.norm(emb, axis=1)
        row.update({
            "embedding_shape": str(tuple(emb.shape)),
            "meta_rows": int(len(meta)),
            "rows_match": bool(len(meta) == emb.shape[0] == expected[name]),
            "norm_mean": float(norms.mean()),
            "norm_min": float(norms.min()),
            "norm_max": float(norms.max()),
        })
    else:
        row.update({
            "embedding_shape": "",
            "meta_rows": 0,
            "rows_match": False,
            "norm_mean": np.nan,
            "norm_min": np.nan,
            "norm_max": np.nan,
        })

    verify_rows.append(row)

df_verify = pd.DataFrame(verify_rows)
verify_csv = REPORT_DIR / f"skeleton_embedding_export_verify_{SPLIT_NAME}.csv"
df_verify.to_csv(verify_csv, index=False)

display(df_verify)

bad = df_verify[df_verify["rows_match"] != True]
if len(bad) > 0:
    raise RuntimeError("Some exported files failed verification. Check df_verify above.")

print("[OK] All exported skeleton embeddings are present and row counts match.")
print("Saved verify report:", verify_csv)

,split,split_tag,expected_rows,npz_exists,csv_exists,pt_exists,npz_path,csv_path,embedding_shape,meta_rows,rows_match,norm_mean,norm_min,norm_max
0,train,train_LT,8139,True,True,True,/media/wadud/DriveUbuntu/GaitRecognition 2.0/r...,/media/wadud/DriveUbuntu/GaitRecognition 2.0/r...,"(8139, 128)",8139,True,1.0,1.0,1.0
1,gallery,gallery_LT,2197,True,True,True,/media/wadud/DriveUbuntu/GaitRecognition 2.0/r...,/media/wadud/DriveUbuntu/GaitRecognition 2.0/r...,"(2197, 128)",2197,True,1.0,1.0,1.0
2,NM,probe_LT_nm,1100,True,True,True,/media/wadud/DriveUbuntu/GaitRecognition 2.0/r...,/media/wadud/DriveUbuntu/GaitRecognition 2.0/r...,"(1100, 128)",1100,True,1.0,1.0,1.0
3,BG,probe_LT_bg,1100,True,True,True,/media/wadud/DriveUbuntu/GaitRecognition 2.0/r...,/media/wadud/DriveUbuntu/GaitRecognition 2.0/r...,"(1100, 128)",1100,True,1.0,1.0,1.0
4,CL,probe_LT_cl,1100,True,True,True,/media/wadud/DriveUbuntu/GaitRecognition 2.0/r...,/media/wadud/DriveUbuntu/GaitRecognition 2.0/r...,"(1100, 128)",1100,True,1.0,1.0,1.0


[OK] All exported skeleton embeddings are present and row counts match.
Saved verify report: /media/wadud/DriveUbuntu/GaitRecognition 2.0/results/fusion_gate/embedding_export_reports/skeleton_embedding_export_verify_LT.csv


In [9]:
# ============================================================
# CELL 9 — Optional sanity evaluation using exported embeddings
# This should be close to your previous skeleton evaluation.
# ============================================================

def load_exported(split_name):
    split_tag = split_tags[split_name]
    npz_path, csv_path, _ = output_paths(split_tag)
    emb = torch.from_numpy(np.load(npz_path)["embeddings"].astype(np.float32))
    meta = pd.read_csv(csv_path, dtype=str)
    return meta, emb

def evaluate_rankk_vectorized(gallery_meta, gallery_emb, probe_meta, probe_emb, condition_name, topk_values=(1, 5, 10), exclude_identical_view=True):
    gallery_subjects = gallery_meta["subject"].astype(str).tolist()
    gallery_views = gallery_meta["view"].astype(str).tolist()

    probe_subjects = probe_meta["subject"].astype(str).tolist()
    probe_views = probe_meta["view"].astype(str).tolist()

    g_emb = F.normalize(gallery_emb.float().cpu(), p=2, dim=1)
    p_emb = F.normalize(probe_emb.float().cpu(), p=2, dim=1)

    # Your skeleton evaluation used Euclidean distance.
    dist = torch.cdist(p_emb, g_emb, p=2)

    if exclude_identical_view:
        for i, p_view in enumerate(probe_views):
            same_view_mask = torch.tensor([g_view == p_view for g_view in gallery_views], dtype=torch.bool)
            dist[i, same_view_mask] = float("inf")

    max_k = max(topk_values)
    topk_indices = torch.topk(dist, k=max_k, dim=1, largest=False).indices.tolist()

    correct_at_k = {k: 0 for k in topk_values}
    rows = []

    for i, idx_list in enumerate(topk_indices):
        true_subject = probe_subjects[i]
        pred_subjects = [gallery_subjects[j] for j in idx_list]
        pred_views = [gallery_views[j] for j in idx_list]
        pred_dists = [float(dist[i, j].item()) for j in idx_list]

        for k in topk_values:
            if true_subject in pred_subjects[:k]:
                correct_at_k[k] += 1

        rows.append({
            "condition": condition_name,
            "probe_index": i,
            "probe_subject": true_subject,
            "probe_view": probe_views[i],
            "probe_seq": str(probe_meta.iloc[i]["seq"]),
            "probe_sample_key": str(probe_meta.iloc[i]["sample_key"]),
            "rank1_correct": bool(pred_subjects[0] == true_subject),
            "rank": next((r + 1 for r, s in enumerate(pred_subjects) if s == true_subject), np.nan),
            "top1_subject": pred_subjects[0],
            "top1_view": pred_views[0],
            "top1_distance": pred_dists[0],
            "top5_subjects": "|".join(pred_subjects[:5]),
            "top5_views": "|".join(pred_views[:5]),
        })

    df = pd.DataFrame(rows)
    summary = {
        "condition": condition_name,
        "num_probe": len(df),
    }
    for k in topk_values:
        summary[f"rank{k}"] = correct_at_k[k] / len(df) if len(df) else 0.0
        summary[f"rank{k}_percent"] = summary[f"rank{k}"] * 100.0

    return summary, df

gallery_meta, gallery_emb = load_exported("gallery")

summary_rows = []
detail_rows = []

for cond in ["NM", "BG", "CL"]:
    probe_meta, probe_emb = load_exported(cond)
    summary, details = evaluate_rankk_vectorized(
        gallery_meta,
        gallery_emb,
        probe_meta,
        probe_emb,
        condition_name=cond,
        topk_values=(1, 5, 10),
        exclude_identical_view=True,
    )
    summary_rows.append(summary)
    detail_rows.append(details)

df_summary = pd.DataFrame(summary_rows)
df_details = pd.concat(detail_rows, ignore_index=True)

summary_csv = REPORT_DIR / f"skeleton_export_sanity_rank_summary_{SPLIT_NAME}.csv"
details_csv = REPORT_DIR / f"skeleton_export_sanity_probe_details_{SPLIT_NAME}.csv"

df_summary.to_csv(summary_csv, index=False)
df_details.to_csv(details_csv, index=False)

display(df_summary)

print("Saved sanity summary:", summary_csv)
print("Saved sanity details :", details_csv)

print("=" * 80)
print("Sanity Rank Results")
print("=" * 80)
for _, row in df_summary.iterrows():
    print(
        f"{row['condition']}: "
        f"Rank-1={row['rank1_percent']:.2f}%, "
        f"Rank-5={row['rank5_percent']:.2f}%, "
        f"Rank-10={row['rank10_percent']:.2f}%"
    )
print("=" * 80)

,condition,num_probe,rank1,rank1_percent,rank5,rank5_percent,rank10,rank10_percent
0,NM,1100,0.950000,95.000000,0.971818,97.181818,0.983636,98.363636
1,BG,1100,0.786364,78.636364,0.888182,88.818182,0.920000,92.000000
2,CL,1100,0.741818,74.181818,0.862727,86.272727,0.913636,91.363636


Saved sanity summary: /media/wadud/DriveUbuntu/GaitRecognition 2.0/results/fusion_gate/embedding_export_reports/skeleton_export_sanity_rank_summary_LT.csv
Saved sanity details : /media/wadud/DriveUbuntu/GaitRecognition 2.0/results/fusion_gate/embedding_export_reports/skeleton_export_sanity_probe_details_LT.csv
Sanity Rank Results
NM: Rank-1=95.00%, Rank-5=97.18%, Rank-10=98.36%
BG: Rank-1=78.64%, Rank-5=88.82%, Rank-10=92.00%
CL: Rank-1=74.18%, Rank-5=86.27%, Rank-10=91.36%


In [10]:
# ============================================================
# CELL 10 — Metadata for notebook 13
# ============================================================

metadata = {
    "purpose": "Skeleton GaitTR embeddings exported for adaptive fusion notebook 13",
    "experiment_dir": str(EXP_DIR),
    "split_name": SPLIT_NAME,
    "checkpoint_path": str(CHECKPOINT_PATH),
    "checkpoint_type": ckpt.get("checkpoint_type", "unknown"),
    "checkpoint_step": ckpt.get("step", "unknown"),
    "eval_mode": EVAL_MODE,
    "seq_len": SEQ_LEN,
    "stride": STRIDE,
    "embedding_dim": int(embedding_dim),
    "cache_dir": str(CACHE_DIR),
    "exported_files": {
        name: {
            "npz": str(output_paths(split_tag)[0]),
            "meta_csv": str(output_paths(split_tag)[1]),
            "pt": str(output_paths(split_tag)[2]),
        }
        for name, split_tag in split_tags.items()
    },
    "next_notebook": "13_train_adaptive_fusion_gate_FULL_FIXED.ipynb",
}

metadata_path = REPORT_DIR / f"skeleton_embedding_export_metadata_{SPLIT_NAME}.json"
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

print("=" * 80)
print("FINAL EXPORT SUMMARY")
print("=" * 80)
print(json.dumps(metadata, indent=2))
print("=" * 80)
print("Saved metadata:", metadata_path)
print()
print("Next step:")
print("Run 13_train_adaptive_fusion_gate_FULL_FIXED.ipynb from the beginning.")

FINAL EXPORT SUMMARY
{
  "purpose": "Skeleton GaitTR embeddings exported for adaptive fusion notebook 13",
  "experiment_dir": "/media/wadud/DriveUbuntu/GaitRecognition 2.0",
  "split_name": "LT",
  "checkpoint_path": "/media/wadud/DriveUbuntu/GaitRecognition 2.0/checkpoints/gaittr_LT_ce_triplet_best_loss.pth",
  "checkpoint_type": "eval_backbone_only",
  "checkpoint_step": 29550,
  "eval_mode": "sliding",
  "seq_len": 60,
  "stride": 30,
  "embedding_dim": 128,
  "cache_dir": "/media/wadud/DriveUbuntu/GaitRecognition 2.0/results/fusion_gate/expert_embedding_cache",
  "exported_files": {
    "train": {
      "npz": "/media/wadud/DriveUbuntu/GaitRecognition 2.0/results/fusion_gate/expert_embedding_cache/skeleton_train_LT_embeddings.npz",
      "meta_csv": "/media/wadud/DriveUbuntu/GaitRecognition 2.0/results/fusion_gate/expert_embedding_cache/skeleton_train_LT_meta.csv",
      "pt": "/media/wadud/DriveUbuntu/GaitRecognition 2.0/results/fusion_gate/expert_embedding_cache/skeleton_train_L

## After this notebook finishes

Now rerun:

```text
13_train_adaptive_fusion_gate_FULL_FIXED.ipynb
```

It should now find:

```text
results/fusion_gate/expert_embedding_cache/skeleton_train_LT_embeddings.npz
results/fusion_gate/expert_embedding_cache/skeleton_gallery_LT_embeddings.npz
results/fusion_gate/expert_embedding_cache/skeleton_probe_LT_nm_embeddings.npz
results/fusion_gate/expert_embedding_cache/skeleton_probe_LT_bg_embeddings.npz
results/fusion_gate/expert_embedding_cache/skeleton_probe_LT_cl_embeddings.npz
```

Then fusion can proceed with paired skeleton + silhouette embeddings.